In [15]:
import weasyprint


-----

WeasyPrint could not import some external libraries. Please carefully follow the installation steps before reporting an issue:
https://doc.courtbouillon.org/weasyprint/stable/first_steps.html#installation
https://doc.courtbouillon.org/weasyprint/stable/first_steps.html#troubleshooting 

-----



OSError: cannot load library 'libgobject-2.0-0': error 0x7e.  Additionally, ctypes.util.find_library() did not manage to locate a library called 'libgobject-2.0-0'

In [14]:
!pip install weasyprint

In [8]:
!pip install reportlab

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ---------------- ----------------------- 0.8/2.0 MB 2.6 MB/s eta 0:00:01
   -------------------------- ------------- 1.3/2.0 MB 2.0 MB/s eta 0:00:01
   -------------------------------- ------- 1.6/2.0 MB 2.1 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 2.1 MB/s  0:00:00


In [3]:
import requests

In [13]:
import weasyprint


-----

WeasyPrint could not import some external libraries. Please carefully follow the installation steps before reporting an issue:
https://doc.courtbouillon.org/weasyprint/stable/first_steps.html#installation
https://doc.courtbouillon.org/weasyprint/stable/first_steps.html#troubleshooting 

-----



OSError: cannot load library 'libgobject-2.0-0': error 0x7e.  Additionally, ctypes.util.find_library() did not manage to locate a library called 'libgobject-2.0-0'

In [11]:
import requests
from bs4 import BeautifulSoup
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

def scrapping2pdf(link, titulo, descripcion):
    nombre_pdf = link.rstrip("/").split("/")[-1]

    # Crear documento PDF
    doc = SimpleDocTemplate(f"../doc_pdf/{nombre_pdf}.pdf", pagesize=letter)
    styles = getSampleStyleSheet()
    story = []

    # Portada
    story.append(Paragraph(f"<b>{titulo}</b>", styles["Title"]))
    story.append(Spacer(1, 12))
    story.append(Paragraph(descripcion, styles["Normal"]))
    story.append(Paragraph(f"Libro recopilado de {link}", styles["Normal"]))
    story.append(Spacer(1, 24))

    # 1. Obtener lista de capítulos
    response = requests.get(link)
    response.encoding = "utf-8"
    soup = BeautifulSoup(response.text, "html.parser")

    chapter_links = []
    for a in soup.select("a[href]"):
        href = a["href"]
        if href.startswith(f"/{nombre_pdf}/") and href != f"/{nombre_pdf}/":
            chapter_links.append("https://basecamp.com" + href)

    chapter_links = sorted(set(chapter_links))

    # 2. Crear índice
    story.append(Paragraph("<b>Index</b>", styles["Heading2"]))
    for i, link in enumerate(chapter_links, 1):
        res = requests.get(link)
        res.encoding = "utf-8"
        chapter_soup = BeautifulSoup(res.text, "html.parser")
        title = chapter_soup.find("h1").get_text(strip=True)
        story.append(Paragraph(f"Chapter {i}. {title}", styles["Normal"]))
    story.append(Spacer(1, 24))

    # 3. Agregar capítulos
    for i, link in enumerate(chapter_links):
        res = requests.get(link)
        res.encoding = "utf-8"
        chapter_soup = BeautifulSoup(res.text, "html.parser")

        title = chapter_soup.find("h1").get_text(strip=True)
        content_elem = chapter_soup.select_one("div.content")

        if not content_elem:
            continue

        # Limpiar contenido
        for unwanted in content_elem.find_all(["footer"], recursive=True):
            unwanted.decompose()

        for p in content_elem.find_all("p"):
            if "We made" in p.get_text() or "Copyright" in p.get_text():
                p.decompose()

        for a in content_elem.find_all("a"):
            if not a.get_text(strip=True):
                a.string = a["href"]

        # Guardar capítulo
        story.append(Paragraph(f"<b>Chapter {i+1}: {title}</b>", styles["Heading2"]))
        story.append(Spacer(1, 12))
        for p in content_elem.find_all("p"):
            text = p.get_text(strip=True)
            if text:
                story.append(Paragraph(text, styles["Normal"]))
                story.append(Spacer(1, 6))

    # 4. Construir PDF
    doc.build(story)
    print(f"✅ Libro {nombre_pdf}.pdf extraído correctamente.")


In [12]:
# Realizamos el llamado a la funcion
link = "https://basecamp.com/gettingreal"
titulo= "Getting Real - Basecamp "
descripcion ="The smarter, faster, easier way to build a successful web application"

scrapping2pdf_fpdf(link, titulo, descripcion)

C:\Users\Angelica\AppData\Local\Temp\ipykernel_23780\2164833504.py:13: DeprecationWarning: Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed
  pdf.set_font("Arial", 'B', 16)
C:\Users\Angelica\AppData\Local\Temp\ipykernel_23780\2164833504.py:16: DeprecationWarning: Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed
  pdf.set_font("Arial", '', 12)


FPDFException: Not enough horizontal space to render a single character